In [0]:
select * from shop_stream.silver.orders;

order_line_id,order_id,customer_id,product_id,quantity,unit_price,order_ts,status,coupon_code
L0000015,O000006,C00332,P0181,1,37.31,2026-06-21T22:50:10.000Z,Cancelled,SAVE10
L0000046,O000029,C00056,P0051,2,83.38,2026-01-14T12:11:45.000Z,Returned,null
L0000047,O000030,C00774,P0176,1,152.07,2026-04-19T04:05:52.000Z,Cancelled,FESTIVE20
L0000061,O000038,C00574,P0112,1,291.53,2026-03-04T09:55:01.000Z,Completed,null
L0000091,O000053,C00284,P0016,1,30.29,2026-04-23T21:49:49.000Z,Completed,null
L0000149,O000088,C00125,P0190,1,40.05,2026-01-13T15:49:09.000Z,Completed,SAVE10
L0000165,O000095,C00758,P0093,1,150.84,2026-04-08T00:50:00.000Z,Completed,WELCOME15
L0000172,O000100,C00939,P0036,2,285.22,2026-01-04T04:59:01.000Z,Completed,SAVE10
L0000182,O000104,C00988,P0028,1,225.44,2026-03-06T17:00:46.000Z,Completed,SAVE10
L0000200,O000115,C00029,P0110,1,27.62,2026-05-02T04:53:23.000Z,Completed,SAVE10


In [0]:
create or replace table shop_stream.gold.daily_revenue
as
with daily_revenue as(
    select order_line_id, 
    order_id, 
    customer_id, 
    product_id, 
    quantity, 
    unit_price, 
    round(unit_price * quantity, 2) as line_revenue,
    order_ts, status, coupon_code 
    from shop_stream.silver.orders
)
select 
date(order_ts) as order_date,
count(distinct(order_id)) as total_orders,
sum(quantity) as total_units_sold,
round(sum(line_revenue),2) as daily_revenue
from daily_revenue 
where status='Completed'
group by date(order_ts)


num_affected_rows,num_inserted_rows


In [0]:
select * from shop_stream.gold.daily_revenue;

order_date,total_orders,total_units_sold,daily_revenue
2026-05-05,33,64,9215.92
2026-06-28,34,81,13279.05
2026-02-28,31,65,9103.39
2026-05-16,39,91,15951.77
2026-02-13,31,67,10448.42
2026-06-11,34,79,11904.83
2026-06-08,25,53,9151.06
2026-05-13,27,69,12621.02
2026-01-24,38,83,13804.05
2026-02-09,23,54,8742.54


In [0]:
select * from shop_stream.silver.products;

product_id,product_name,category,unit_price,unit_cost
P0001,Wireless Earbuds,electronics,69.01,45.86
P0002,Wireless Earbuds v2,electronics,244.41,127.05
P0003,Wireless Earbuds v3,electronics,41.1,24.58
P0004,Wireless Earbuds v4,electronics,89.69,48.74
P0005,Wireless Earbuds v5,electronics,281.95,211.29
P0006,Mechanical Keyboard,electronics,229.55,105.48
P0007,Mechanical Keyboard v2,electronics,27.92,18.85
P0008,Mechanical Keyboard v3,electronics,40.16,21.28
P0009,Mechanical Keyboard v4,electronics,105.65,64.76
P0010,Mechanical Keyboard v5,electronics,47.69,34.28


In [0]:
create or replace table shop_stream.gold.revenue_category as
with products as(
    select 
    product_id,
    category,
    unit_price,
    unit_cost,
    round(unit_price - unit_cost,2) as unit_margin
    from shop_stream.silver.products
)
,sales as(
    select order_line_id, 
    order_id, 
    customer_id, 
    product_id, 
    quantity, 
    unit_price, 
    round(unit_price * quantity, 2) as line_revenue,
    order_ts, status, coupon_code 
    from shop_stream.silver.orders
)

select 
  p.category,
  COUNT(DISTINCT o.order_id) AS orders,
  SUM(o.quantity) AS units_sold,
  ROUND(SUM(o.line_revenue), 2) AS revenue,
  ROUND(SUM(o.quantity * p.unit_margin), 2) AS gross_margin
FROM sales o
JOIN products p USING (product_id)
WHERE o.status = 'Completed'
GROUP BY p.category;


    

num_affected_rows,num_inserted_rows


In [0]:
select * from shop_stream.gold.revenue_category;

category,orders,units_sold,revenue,gross_margin
toys,915,1326,233182.84,103171.13
grocery,902,1352,220361.17,85506.2
home-kitchen,1291,1935,284432.61,112616.01
fashion,1052,1556,268969.44,107491.71
electronics,1152,1674,243726.66,104641.39
beauty,918,1306,187268.73,71379.89
books,1058,1533,261403.86,92158.44
fitness,957,1411,258136.76,108812.66


In [0]:
select * from shop_stream.silver.customers

customer_id,name,email,city,country,signup_date,signup_channel
C00001,Divya Sharma,divya.sharma1@example.com,Singapore,SG,2025-05-08,paid_search
C00002,Arjun Kumar,arjun.kumar2@example.com,Berlin,DE,2025-12-18,organic
C00003,Amelia Patel,amelia.patel3@example.com,Mumbai,IN,2025-02-07,paid_search
C00004,Priya Anderson,priya.anderson4@example.com,Hyderabad,IN,2025-12-21,email
C00005,Mason Mehta,mason.mehta5@example.com,San Francisco,US,2025-10-09,organic
C00006,Rohan Davis,rohan.davis6@example.com,Chennai,IN,2025-05-05,paid_search
C00007,Ava Kumar,ava.kumar7@example.com,Bengaluru,IN,2025-07-04,social
C00008,Ethan Lopez,ethan.lopez8@example.com,Pune,IN,2025-01-24,referral
C00009,Divya Miller,divya.miller9@example.com,Bengaluru,IN,2025-09-10,email
C00010,Sophia Martinez,sophia.martinez10@example.com,Hyderabad,IN,2025-12-03,organic


In [0]:
create or replace table shop_stream.gold.customer_lifetime
as
select 
  c.customer_id,
  c.name,
  c.country,
  c.signup_channel,
  COUNT(DISTINCT o.order_id) AS lifetime_orders,
  ROUND(SUM(round(o.quantity * o.unit_price, 2)), 2) AS lifetime_revenue
FROM shop_stream.silver.orders o
JOIN shop_stream.silver.customers c USING (customer_id)
WHERE o.status = 'Completed'
GROUP BY c.customer_id, c.name, c.country, c.signup_channel;

num_affected_rows,num_inserted_rows
